# 03 — Train configured models

**Purpose.** Fit the configured logistic-regression and XGBoost models using development admissions only.

**Training policy**
- Development data = rows labelled `train` or `validation`.
- The temporal `test` period is never used for fitting.
- Unknown or unresolved outcomes are excluded by the reusable target encoder.
- Imputation, encoding, and scaling are fitted inside each model pipeline.
- Each saved model bundle includes its feature list, target name, and training metadata.


In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from pathlib import Path
import joblib
import yaml
import pandas as pd
from IPython.display import display

from icare_risk.data.audit import load_admission_file
from icare_risk.data.feature_sets import resolve_feature_set
from icare_risk.models.training import encode_binary_target, fit_model

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

print(f'Configuration: {CONFIG_PATH}')

dataset_path = config['outputs']['processed_dataset']
df = load_admission_file(dataset_path)
columns = config['columns']
target = columns['target']
target_mapping = config.get(
    'target_mapping',
    config.get('analysis', {}).get('target_mapping', {}),
)
development_splits = ['train', 'validation']
development = df[df['split'].isin(development_splits)].copy()
test = df[df['split'].eq('test')].copy()

if target not in development.columns:
    raise KeyError(f'Target column {target!r} is not present in the dataset.')

encoded = encode_binary_target(
    development[target],
    positive_values=target_mapping.get('positive_values', []),
    negative_values=target_mapping.get('negative_values', []),
)

print(f'Dataset: {dataset_path}')
print(f'Rows and columns: {df.shape}')
print(f'Development splits: {development_splits}')
print(f'Development admissions: {len(development):,}')
print(f'Temporal test admissions: {len(test):,}')
print(f'Development positive outcomes: {int((encoded == 1).sum()):,}')
print(f'Development negative outcomes: {int((encoded == 0).sum()):,}')
print(f'Excluded unknown/missing outcomes: {int(encoded.isna().sum()):,}')

Configuration: ../src/icare_risk/config/icare/analysis.yaml
Dataset: data/processed/admissions_with_split.parquet
Rows and columns: (10000, 187)
Development splits: ['train', 'validation']
Development admissions: 7,517
Temporal test admissions: 2,483
Development positive outcomes: 1,465
Development negative outcomes: 6,052
Excluded unknown/missing outcomes: 0


In [30]:
models = {}
feature_map = {}
training_rows = []

for model_cfg in config.get('models', []):
    name = model_cfg['name']
    feature_set_name = model_cfg['feature_set']
    if feature_set_name not in config.get('feature_sets', {}):
        raise KeyError(f'Feature set {feature_set_name!r} is not defined in the configuration.')

    features = resolve_feature_set(df,
        config['feature_sets'][feature_set_name],
        exclude_columns=[target, 'split'],
    )
    if not features:
        raise ValueError(f'Model {name!r} resolved to zero predictor features.')

    print(f'\nTraining {name}')
    print(f'  Type: {model_cfg["type"]}')
    print(f'  Feature set: {feature_set_name}')
    print(f'  Number of features: {len(features)}')
    print(f'  Parameters: {model_cfg.get("params", {})}')
    #print(f'  First features: {features[:10]}')

    fitted_model = fit_model(
        df=development,
        features=features,
        target=target,
        model_type=model_cfg['type'],
        params=model_cfg.get('params', {}),
        scale_numeric=model_cfg.get('scale_numeric', False),
        target_mapping=target_mapping,
    )
    models[name] = fitted_model
    feature_map[name] = features

    training_rows.append({
        'model': name,
        'model_type': model_cfg['type'],
        'feature_set': feature_set_name,
        'n_features': len(features),
        'n_development_rows': len(development),
        'n_test_rows': len(test),
        'n_positive_development': int((encoded == 1).sum()),
        'n_negative_development': int((encoded == 0).sum()),
    })

training_summary = pd.DataFrame(training_rows)
display(training_summary)



Training lr_kim_components
  Type: logistic_regression
  Feature set: kim_components
  Number of features: 5
  Parameters: {'C': 1.0}

Training lr_all_score_values
  Type: logistic_regression
  Feature set: all_score_values
  Number of features: 6
  Parameters: {'C': 1.0}

Training lr_clinical_baseline
  Type: logistic_regression
  Feature set: clinical_baseline
  Number of features: 16
  Parameters: {'C': 1.0}

Training xgb_kim_components
  Type: xgboost
  Feature set: kim_components
  Number of features: 5
  Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}

Training xgb_all_score_values
  Type: xgboost
  Feature set: all_score_values
  Number of features: 6
  Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}

Training xgb_clinical_baseline
  Type: xgboost
  Feature set: clinical_baseline
  Number of features: 16
  Parameters: {'n_estimators': 300, 'max

,model,model_type,feature_set,n_features,n_development_rows,n_test_rows,n_positive_development,n_negative_development
0,lr_kim_components,logistic_regression,kim_components,5,7517,2483,1465,6052
1,lr_all_score_values,logistic_regression,all_score_values,6,7517,2483,1465,6052
2,lr_clinical_baseline,logistic_regression,clinical_baseline,16,7517,2483,1465,6052
3,xgb_kim_components,xgboost,kim_components,5,7517,2483,1465,6052
4,xgb_all_score_values,xgboost,all_score_values,6,7517,2483,1465,6052
5,xgb_clinical_baseline,xgboost,clinical_baseline,16,7517,2483,1465,6052


In [33]:
model_directory = Path(config['outputs']['model_directory'])
model_directory.mkdir(parents=True, exist_ok=True)
for name, model in models.items():
    bundle = {
        'model': model,
        'features': feature_map[name],
        'target': target
    }
    output_path = model_directory / f'{name}.joblib'
    joblib.dump(bundle, output_path)
print(f'Saved {len(models)} fitted models to {model_directory}')

summary_path = model_directory / 'training_summary.csv'
training_summary.to_csv(summary_path, index=False)
print(f'Saved training summary: {summary_path}')

Saved 6 fitted models to models/fitted
Saved training summary: models/fitted/training_summary.csv
